In [6]:
# Combined Preprocessing Notebook

import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

# Handling Missing Data
# Source: IT24102297_Handling_missing_data.ipynb
# Assigned to: IT24102297

start_time = time.time()
print("Starting Member 1 Notebook: Handling Missing Data at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset 
df = pd.read_csv('../data/raw/Students Performance Dataset.csv')
print("Dataset loaded. Shape:", df.shape)

# Check missing values
print("\nMissing values before imputation:")
print(df.isnull().sum())
# Technique: Impute with mean for numerical, mode for categorical
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
cat_cols = df.select_dtypes(include=['object']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])
print("\nMissing values after imputation:")
print(df.isnull().sum().sum(), "total missing values remaining (should be 0)")

# Simple Visualization: Histogram of Total_Score
plt.figure()
plt.hist(df['Total_Score'], bins=10)
plt.title('Member 1 (IT24102297): Distribution of Total Score')
plt.xlabel('Total Score')
plt.ylabel('Count')
plt.savefig('../results/eda_visualizations/eda_IT24102297_histogram.png')
plt.close()

# Save output 
df.to_csv('../results/outputs/imputed_data.csv', index=False)
print("Saved: imputed_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 1 Task Complete.")

# Encoding Categorical Variables
# Source: IT24102257_Encoding_categorical_variables.ipynb
# Assigned to: IT24102257

start_time = time.time()
print("Starting Member 2 Notebook: Encoding Categorical Variables at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset 
df = pd.read_csv('../results/outputs/imputed_data.csv')
print("Dataset loaded. Shape:", df.shape)

# One-Hot Encoding
df = pd.get_dummies(df, columns=['Gender', 'Department'], drop_first=False)
print("One-Hot Encoded Gender and Department successfully.\n")
print(df.head(5))
# Label Encoding
grade_order = {'A':4, 'B':3, 'C':2, 'D':1, 'F':0}
df['Grade'] = df['Grade'].map(grade_order)
education_order = {'High School':0, "Bachelor's":1, "Master's":2, 'PhD':3}
df['Parent_Education_Level'] = df['Parent_Education_Level'].map(education_order)
income_order = {'Low':0, 'Medium':1, 'High':2}
df['Family_Income_Level'] = df['Family_Income_Level'].map(income_order)
df['Extracurricular_Activities'] = df['Extracurricular_Activities'].map({'No':0, 'Yes':1})
df['Internet_Access_at_Home'] = df['Internet_Access_at_Home'].map({'No':0, 'Yes':1})
print(df.head(5))

# Identify numerical columns for correlation heatmap
num_cols = df.select_dtypes(include=['float64', 'int64', 'int']).columns


# Plot correlation heatmap
plt.figure(figsize=(12, 10))
plt.imshow(df[num_cols].corr(), cmap='hot', interpolation='nearest')
plt.colorbar()
plt.title('Member 2 (IT24102257): Correlation Heatmap')
plt.xticks(ticks=range(len(num_cols)), labels=num_cols, rotation=90)
plt.yticks(ticks=range(len(num_cols)), labels=num_cols)
plt.tight_layout()
plt.savefig('../results/eda_visualizations/eda_IT24102257_heatmap.png')
plt.close()


# Save output 
df.to_csv('../results/outputs/encoded_data.csv', index=False)
print("Saved: encoded_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 2 Task Complete.")

# Feature Creation
# Source: IT24102306_Feature_Creation.ipynb
# Assigned to: IT24102306

start_time = time.time()
print("Starting Member 3 Notebook: Feature Creation at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset 
df = pd.read_csv('../results/outputs/encoded_data.csv')
print("Dataset loaded. Shape:", df.shape)


# Create new features
# 1. Average_Score: Mean of academic performance metrics
df['Average_Score'] = df[['Midterm_Score', 'Final_Score', 'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score', 'Projects_Score']].mean(axis=1)
# 2. Study_Efficiency: Total_Score / Study_Hours_per_Week (handle zero division)
df['Study_Efficiency'] = df.apply(lambda x: x['Total_Score'] / x['Study_Hours_per_Week'] if x['Study_Hours_per_Week'] != 0 else 0, axis=1)
# 3. Stress_Sleep_Ratio: Stress_Level / Sleep_Hours_per_Night (handle zero division)
df['Stress_Sleep_Ratio'] = df.apply(lambda x: x['Stress_Level (1-10)'] / x['Sleep_Hours_per_Night'] if x['Sleep_Hours_per_Night'] != 0 else 0, axis=1)
print("Created new features: Average_Score, Study_Efficiency, Stress_Sleep_Ratio")


# Simple Visualization: Histogram of Average_Score
plt.figure()
plt.hist(df['Average_Score'], bins=20, edgecolor='k')
plt.title('Member 3 (IT24102306): Distribution of Average Score')
plt.xlabel('Average Score')
plt.ylabel('Frequency')
plt.savefig('../results/eda_visualizations/eda_IT24102306_histogram.png')
plt.close()

# Save output 
df.to_csv('../results/outputs/created_features_data.csv', index=False)
print("Saved: created_features_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 3 Task Complete.")

# Outlier Removal
# Source: IT24102308_Outlier_removal.ipynb
# Assigned to: IT24102308
# Start timer
start_time = time.time()
print("Starting Member 4 Notebook: Outlier Removal at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset with relative path
df = pd.read_csv('../results/outputs/created_features_data.csv')
print("Dataset loaded. Shape:", df.shape)
# Technique: IQR for numerical columns
num_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower) & (df[col] <= upper)]
print("Dataset shape after outlier removal:", df.shape)
# Visualization: Boxplot for numerical columns
plt.figure(figsize=(10, 6))
df[num_cols].boxplot()
plt.title('Member 4 (IT24102308): Boxplot of Numerical Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../results/eda_visualizations/eda_IT24102308_boxplot.png')
plt.close()
# Save output with relative path
df.to_csv('../results/outputs/cleaned_outliers_data.csv', index=False)
print("Saved: cleaned_outliers_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 4 Task Complete.")

# Normalization & Scaling
# Source: IT24102225_Normalization_scaling.ipynb
# Assigned to: IT24102225

start_time = time.time()
print("Starting Member 5 Notebook: Normalization & Scaling at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset 
df = pd.read_csv('../results/outputs/cleaned_outliers_data.csv')
print("Dataset loaded. Shape:", df.shape)

# Identify numerical columns
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
# Apply Standardization (Z-score scaling)
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
print("StandardScaler applied successfully.\n")
print(df.head(5))


# Simple Visualization: Histogram of Total_Score after scaling
plt.figure()
plt.hist(df['Total_Score'], bins=20, edgecolor='k')
plt.title('Member 5 (IT24102225): Distribution of Total Score after Scaling')
plt.xlabel('Total Score (scaled)')
plt.ylabel('Frequency')
plt.savefig('../results/eda_visualizations/eda_IT24102225_hist.png')
plt.close()

# Save output 
df.to_csv('../results/outputs/scaled_data.csv', index=False)
print("Saved: scaled_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 5 Task Complete.")

# Feature Engineering (Feature Selection + Dimension Reduction)
# Source: IT24102298_Feature_Engineering.ipynb
# Assigned to: IT24102298

start_time = time.time()
print("Starting Member 6 Notebook: Feature Engineering (Feature Selection + Dimension Reduction) at", time.strftime("%H:%M:%S", time.localtime()))
# Load dataset
df = pd.read_csv('../results/outputs/scaled_data.csv')
print("Dataset loaded. Shape:", df.shape)

# Drop identifier columns (non-numeric, e.g., Student_ID, names, email) to avoid conversion errors
df = df.drop(['Student_ID', 'First_Name', 'Last_Name', 'Email'], axis=1, errors='ignore')
print("Dropped identifiers. Updated shape:", df.shape)

# Feature Selection: SelectKBest
X = df.drop(['Total_Score', 'Grade'], axis=1)
y = df['Total_Score']
selector = SelectKBest(score_func=f_regression, k=8)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
print("Selected features:", selected_features)
# Create DataFrame with selected features
df_selected = pd.DataFrame(X_selected, columns=selected_features, index=df.index)
df_selected['Total_Score'] = y
print("Feature selection completed. Shape:", df_selected.shape)

# Dimension Reduction: PCA
numeric_cols = df_selected.select_dtypes(include=['float64', 'int64']).columns
X = df_selected[numeric_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Data scaled for PCA. Shape:", X_scaled.shape)
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)
explained_variance = pca.explained_variance_ratio_.sum()
print(f"PCA explained variance: {explained_variance:.2f}")
X_reduced = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(5)], index=df_selected.index)
print("Dimension reduction completed.")
# Merge reduced components back with Total_Score
df_pca = pd.concat([df_selected[['Total_Score']], X_reduced], axis=1)
print("Merged PCA components with Total_Score. New shape:", df_pca.shape)


# EDA Visualization: Scatter Plot of PC1 vs PC2
print("\nEDA - Scatter Plot of Principal Components")
df_pca['At_Risk'] = df_pca['Total_Score'].apply(lambda s: 1 if s < 60 else 0)
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df_pca, x='PC1', y='PC2', hue='At_Risk', palette='husl', s=100)
plt.title('Member 6 (IT24102298): Scatter Plot of PC1 vs PC2 by At-Risk Status')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.savefig('../results/eda_visualizations/eda_IT24102298_scatterplot.png')
plt.close()
print("Saved scatter plot. Interpretation: Shows relationship between PC1 and PC2, colored by at-risk status to identify potential clusters.")

# Save output 
df_pca.to_csv('../results/outputs/final_preprocessed_data.csv', index=False)
print("Final preprocessed data saved: final_preprocessed_data.csv")
print(f"\nRuntime: {time.time() - start_time:.2f} seconds")
print("Member 6 Task Complete.")

Starting Member 1 Notebook: Handling Missing Data at 20:39:00
Dataset loaded. Shape: (5000, 23)

Missing values before imputation:
Student_ID                       0
First_Name                       0
Last_Name                        0
Email                            0
Gender                           0
Age                              0
Department                       0
Attendance (%)                   0
Midterm_Score                    0
Final_Score                      0
Assignments_Avg                  0
Quizzes_Avg                      0
Participation_Score              0
Projects_Score                   0
Total_Score                      0
Grade                            0
Study_Hours_per_Week             0
Extracurricular_Activities       0
Internet_Access_at_Home          0
Parent_Education_Level        1025
Family_Income_Level              0
Stress_Level (1-10)              0
Sleep_Hours_per_Night            0
dtype: int64

Missing values after imputation:
0 total missing v